In [ ]:
---
layout: post
title: Platformer/Gravity lesson
courses: csse
codemirror: true
microblog: true
permalink: /js/platformergame
author: Jasan, Krish, Aarnav
mermaid: false
---



### What is Gravity in a Platformer


hi

Imagine jumping in a game and floating forever without coming down. That would feel completely broken. Gravity is what pulls your character back down and makes movement feel real. Gravity in a platformer is a simple system that constantly pulls the player downward. It makes jumping, falling, and landing possible.

### Try It in a Real Level

This runner loads **GameLevel3: Crater Falls** so you can feel gravity, jumping, moving platforms, and collision timing in an actual platformer level.


In [ ]:
%%js

// GAME_RUNNER: Play Crater Falls and test gravity, jumps, and platform collisions. Use WASD or arrow keys to move and jump. | hide_edit: true, width: 100%, height: 500px

import { GameControl } from '/assets/js/GameEnginev1.1/essentials/Imports.js';
import GameLevel3 from '/assets/js/projects/AstronautPlatformergame/levels/GameLevel3.js';

export const gameLevelClasses = [GameLevel3];
export { GameControl };


### Example #1: Basic Gravity

One example is how a player falls when they walk off a platform. The game keeps increasing the downward speed every frame.


In [ ]:
%%js


let y = 100;
let vy = 0;
const gravity = 0.5;

function update() {
    vy += gravity;   // gravity pulls down
    y += vy;         // apply movement

    console.log("Player height:", y);
}

// Simulate frames
for (let i = 0; i < 5; i++) {
    update();
}


In [ ]:
%%js

// GAME_RUNNER: Gravity
import GameEnvBackground from '/assets/js/GameEnginev1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1/essentials/Player.js';
import Npc from '/assets/js/GameEnginev1/essentials/Npc.js';
import Barrier from '/assets/js/GameEnginev1/essentials/Barrier.js';

class GameLevelCustom {
    constructor(gameEnv) {
        const path = gameEnv.path;
        const width = gameEnv.innerWidth;
        const height = gameEnv.innerHeight;

        const bgData = {
            name: "custom_bg",
            src: path + "/images/gamebuilder/bg/alien_planet.jpg",
            pixels: { height: 772, width: 1134 }
        };

        const playerData = {
            id: 'playerData',
            src: path + "/images/gamebuilder/sprites/astro.png",
            SCALE_FACTOR: 5,
            STEP_FACTOR: 1000,
            ANIMATION_RATE: 50,
            INIT_POSITION: { x: 100, y: 52 },
            pixels: { height: 770, width: 513 },
            orientation: { rows: 4, columns: 4 },
            down: { row: 0, start: 0, columns: 3 },
            downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI/16 },
            downLeft: { row: 0, start: 0, columns: 3, rotate: -Math.PI/16 },
            left: { row: 2, start: 0, columns: 3 },
            right: { row: 1, start: 0, columns: 3 },
            up: { row: 3, start: 0, columns: 3 },
            upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI/16 },
            upRight: { row: 3, start: 0, columns: 3, rotate: -Math.PI/16 },
            hitbox: { widthPercentage: 0, heightPercentage: 0 },
            keypress: { up: 87, left: 65, down: 83, right: 68 },
            GRAVITY: false
        };

        this.classes = [
            { class: GameEnvBackground, data: bgData },
            { class: Player, data: playerData }
        ];

        // ── Gravity — mirrors the code runner example exactly ─────────────────
        // let y = player.position.y  (the engine owns this)
        // let vy = 0
        // const gravity = 0.5
        // every frame: vy += gravity;  y += vy;
        this.initialize = () => {
            const player = gameEnv.gameObjects.find(obj => obj instanceof Player);
            if (!player) return;

            let vy = 0;              // same as code runner example
            const gravity = 0.5;    // same as code runner example

            const _originalUpdate = player.update.bind(player);

            player.update = function () {
                // gravity pulls down (vy grows each frame)
                vy += gravity;

                // y += vy  — apply to position, not velocity so Player.js can't override it
                this.velocity.y = 0;
                this.position.y += vy;

                // floor clamp
                const floor = gameEnv.innerHeight - this.height;
                if (this.position.y >= floor) {
                    this.position.y = floor;
                    vy = 0;
                }

                // ceiling clamp
                if (this.position.y < 0) {
                    this.position.y = 0;
                    vy = 0;
                }

                // run original update for drawing, collision, and left/right movement
                _originalUpdate();

                // zero velocity.y again so Player.js move() doesn't undo gravity
                this.velocity.y = 0;
            };
        };

        /* BUILDER_ONLY_START */
        try {
            setTimeout(() => {
                try {
                    const objs = Array.isArray(gameEnv?.gameObjects) ? gameEnv.gameObjects : [];
                    const summary = objs.map(o => ({ cls: o?.constructor?.name || 'Unknown', id: o?.canvas?.id || '', z: o?.canvas?.style?.zIndex || '' }));
                    if (window && window.parent) window.parent.postMessage({ type: 'rpg:objects', summary }, '*');
                } catch (_) {}
            }, 250);
        } catch (_) {}
        try {
            if (window && window.parent) {
                try {
                    const rect = (gameEnv && gameEnv.container && gameEnv.container.getBoundingClientRect) ? gameEnv.container.getBoundingClientRect() : { top: gameEnv.top || 0, left: 0 };
                    window.parent.postMessage({ type: 'rpg:env-metrics', top: rect.top, left: rect.left }, '*');
                } catch (_) {
                    try { window.parent.postMessage({ type: 'rpg:env-metrics', top: gameEnv.top, left: 0 }, '*'); } catch (__){ }
                }
            }
        } catch (_) {}
        try {
            window.addEventListener('message', (e) => {
                if (!e || !e.data) return;
                if (e.data.type === 'rpg:toggle-walls') {
                    const show = !!e.data.visible;
                    if (Array.isArray(gameEnv?.gameObjects)) {
                        for (const obj of gameEnv.gameObjects) {
                            if (obj instanceof Barrier) {
                                obj.visible = show;
                            }
                        }
                    }
                } else if (e.data.type === 'rpg:set-drawn-barriers') {
                    const arr = Array.isArray(e.data.barriers) ? e.data.barriers : [];
                    window.__overlayBarriers = window.__overlayBarriers || [];
                    try {
                        for (const ob of window.__overlayBarriers) {
                            if (ob && typeof ob.destroy === 'function') ob.destroy();
                        }
                    } catch (_) {}
                    window.__overlayBarriers = [];
                    for (const bd of arr) {
                        try {
                            const data = {
                                id: bd.id,
                                x: bd.x,
                                y: bd.y,
                                width: bd.width,
                                height: bd.height,
                                visible: !!bd.visible,
                                hitbox: { widthPercentage: 0.0, heightPercentage: 0.0 },
                                fromOverlay: true
                            };
                            const bobj = new Barrier(data, gameEnv);
                            gameEnv.gameObjects.push(bobj);
                            window.__overlayBarriers.push(bobj);
                        } catch (_) {}
                    }
                }
            });
        } catch (_) {}
        /* BUILDER_ONLY_END */
    }
}

export const gameLevelClasses = [GameLevelCustom];

### What's happening:
The player starts with no movement, but gravity increases velocity each frame, causing faster and faster falling.


### Example #2: Jumping

The player should be able to jump up and then come back down naturally.


In [ ]:
%%js

let y = 300;
let vy = 0;
const gravity = 0.5;

function jump() {
    // Add code here
}

function update() {
    vy += gravity;
    y += vy;

    console.log("y:", y);
}

jump();
for (let i = 0; i < 5; i++) {
    update();
}


In [ ]:
%%js

//GAME_RUNNER: Jumping

import GameEnvBackground from '/assets/js/GameEnginev1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1/essentials/Player.js';
import Npc from '/assets/js/GameEnginev1/essentials/Npc.js';
import Barrier from '/assets/js/GameEnginev1/essentials/Barrier.js';

class GameLevelCustom {
    constructor(gameEnv) {
        const path = gameEnv.path;
        const width = gameEnv.innerWidth;
        const height = gameEnv.innerHeight;

        const bgData = {
            name: "custom_bg",
            src: path + "/images/gamebuilder/bg/alien_planet.jpg",
            pixels: { height: 772, width: 1134 }
        };

        const playerData = {
            id: 'playerData',
            src: path + "/images/gamebuilder/sprites/astro.png",
            SCALE_FACTOR: 5,
            STEP_FACTOR: 1000,
            ANIMATION_RATE: 50,
            INIT_POSITION: { x: 100, y: 52 },
            pixels: { height: 770, width: 513 },
            orientation: { rows: 4, columns: 4 },
            down: { row: 0, start: 0, columns: 3 },
            downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI/16 },
            downLeft: { row: 0, start: 0, columns: 3, rotate: -Math.PI/16 },
            left: { row: 2, start: 0, columns: 3 },
            right: { row: 1, start: 0, columns: 3 },
            up: { row: 3, start: 0, columns: 3 },
            upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI/16 },
            upRight: { row: 3, start: 0, columns: 3, rotate: -Math.PI/16 },
            hitbox: { widthPercentage: 0, heightPercentage: 0 },
            keypress: { up: 87, left: 65, down: 83, right: 68 },
            GRAVITY: false
        };

        this.classes = [
            { class: GameEnvBackground, data: bgData },
            { class: Player, data: playerData }
        ];

        // ── Gravity — mirrors the code runner example exactly ─────────────────
        // let y = player.position.y  (the engine owns this)
        // let vy = 0
        // const gravity = 0.5
        // every frame: vy += gravity;  y += vy;
        this.initialize = () => {
            const player = gameEnv.gameObjects.find(obj => obj instanceof Player);
            if (!player) return;

            let vy = 0;              // same as code runner example
            const gravity = 0.5;    // same as code runner example

            // jump() — gives an instant upward kick, just like the code runner example
            function jump() {
                vy = -10;  // negative = upward on screen
            }

            const _originalUpdate = player.update.bind(player);

            player.update = function () {
                // W key triggers jump() on fresh press
                const wDown = this.pressedKeys?.[87];
                if (wDown && !this._wWasDown) {
                    jump();
                }
                this._wWasDown = !!wDown;

                // gravity pulls down (vy grows each frame)
                vy += gravity;

                // y += vy  — apply to position, not velocity so Player.js can't override it
                this.velocity.y = 0;
                this.position.y += vy;

                // floor clamp
                const floor = gameEnv.innerHeight - this.height;
                if (this.position.y >= floor) {
                    this.position.y = floor;
                    vy = 0;
                }

                // ceiling clamp
                if (this.position.y < 0) {
                    this.position.y = 0;
                    vy = 0;
                }

                // run original update for drawing, collision, and left/right movement
                _originalUpdate();

                // zero velocity.y again so Player.js move() doesn't undo gravity
                this.velocity.y = 0;
            };
        };

        /* BUILDER_ONLY_START */
        try {
            setTimeout(() => {
                try {
                    const objs = Array.isArray(gameEnv?.gameObjects) ? gameEnv.gameObjects : [];
                    const summary = objs.map(o => ({ cls: o?.constructor?.name || 'Unknown', id: o?.canvas?.id || '', z: o?.canvas?.style?.zIndex || '' }));
                    if (window && window.parent) window.parent.postMessage({ type: 'rpg:objects', summary }, '*');
                } catch (_) {}
            }, 250);
        } catch (_) {}
        try {
            if (window && window.parent) {
                try {
                    const rect = (gameEnv && gameEnv.container && gameEnv.container.getBoundingClientRect) ? gameEnv.container.getBoundingClientRect() : { top: gameEnv.top || 0, left: 0 };
                    window.parent.postMessage({ type: 'rpg:env-metrics', top: rect.top, left: rect.left }, '*');
                } catch (_) {
                    try { window.parent.postMessage({ type: 'rpg:env-metrics', top: gameEnv.top, left: 0 }, '*'); } catch (__){ }
                }
            }
        } catch (_) {}
        try {
            window.addEventListener('message', (e) => {
                if (!e || !e.data) return;
                if (e.data.type === 'rpg:toggle-walls') {
                    const show = !!e.data.visible;
                    if (Array.isArray(gameEnv?.gameObjects)) {
                        for (const obj of gameEnv.gameObjects) {
                            if (obj instanceof Barrier) {
                                obj.visible = show;
                            }
                        }
                    }
                } else if (e.data.type === 'rpg:set-drawn-barriers') {
                    const arr = Array.isArray(e.data.barriers) ? e.data.barriers : [];
                    window.__overlayBarriers = window.__overlayBarriers || [];
                    try {
                        for (const ob of window.__overlayBarriers) {
                            if (ob && typeof ob.destroy === 'function') ob.destroy();
                        }
                    } catch (_) {}
                    window.__overlayBarriers = [];
                    for (const bd of arr) {
                        try {
                            const data = {
                                id: bd.id,
                                x: bd.x,
                                y: bd.y,
                                width: bd.width,
                                height: bd.height,
                                visible: !!bd.visible,
                                hitbox: { widthPercentage: 0.0, heightPercentage: 0.0 },
                                fromOverlay: true
                            };
                            const bobj = new Barrier(data, gameEnv);
                            gameEnv.gameObjects.push(bobj);
                            window.__overlayBarriers.push(bobj);
                        } catch (_) {}
                    }
                }
            });
        } catch (_) {}
        /* BUILDER_ONLY_END */
    }
}

export const gameLevelClasses = [GameLevelCustom];

### Gravity with Game Mechanics

In real games, gravity is combined with limits and collisions to make gameplay smooth.

### Example #3: Limiting Fall Speed


In [ ]:
%%js

let vy = 0;
const gravity = 0.5;
const maxFall = 10;

function update() {
    vy = Math.min(vy + gravity, maxFall);
    console.log("Velocity:", vy);
}

for (let i = 0; i < 10; i++) {
    update();
}


In [ ]:
%%js

//GAME_RUNNER: Terminal Velocity

import GameEnvBackground from '/assets/js/GameEnginev1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1/essentials/Player.js';
import Npc from '/assets/js/GameEnginev1/essentials/Npc.js';
import Barrier from '/assets/js/GameEnginev1/essentials/Barrier.js';

class GameLevelCustom {
    constructor(gameEnv) {
        const path = gameEnv.path;
        const width = gameEnv.innerWidth;
        const height = gameEnv.innerHeight;

        const bgData = {
            name: "custom_bg",
            src: path + "/images/gamebuilder/bg/alien_planet.jpg",
            pixels: { height: 772, width: 1134 }
        };

        const playerData = {
            id: 'playerData',
            src: path + "/images/gamebuilder/sprites/astro.png",
            SCALE_FACTOR: 5,
            STEP_FACTOR: 1000,
            ANIMATION_RATE: 50,
            INIT_POSITION: { x: 100, y: 52 },
            pixels: { height: 770, width: 513 },
            orientation: { rows: 4, columns: 4 },
            down: { row: 0, start: 0, columns: 3 },
            downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI/16 },
            downLeft: { row: 0, start: 0, columns: 3, rotate: -Math.PI/16 },
            left: { row: 2, start: 0, columns: 3 },
            right: { row: 1, start: 0, columns: 3 },
            up: { row: 3, start: 0, columns: 3 },
            upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI/16 },
            upRight: { row: 3, start: 0, columns: 3, rotate: -Math.PI/16 },
            hitbox: { widthPercentage: 0, heightPercentage: 0 },
            keypress: { up: 87, left: 65, down: 83, right: 68 },
            GRAVITY: false
        };

        const platformData = {
            id: 'platform',
            x: 0,
            y: height * 0.9,
            width: width,
            height: 20,
            visible: true,
            color: 'rgba(180, 120, 40, 0.9)',
            hitbox: { widthPercentage: 0.0, heightPercentage: 0.0 },
            fromOverlay: true
        };

        this.classes = [
            { class: GameEnvBackground, data: bgData },
            { class: Player, data: playerData },
            { class: Barrier, data: platformData }
        ];

        this.initialize = () => {
            const player = gameEnv.gameObjects.find(obj => obj instanceof Player);
            if (!player) return;

            let vy = 0;
            const gravity = 0.5;

            function jump() {
                vy = -10;
            }

            const _originalUpdate = player.update.bind(player);

            player.update = function () {
                const wDown = this.pressedKeys?.[87];
                if (wDown && !this._wWasDown) {
                    jump();
                }
                this._wWasDown = !!wDown;

                const maxFall = 10;
                vy = Math.min(vy + gravity, maxFall);

                this.velocity.y = 0;
                this.position.y += vy;

                const floor = gameEnv.innerHeight - this.height;
                if (this.position.y >= floor) {
                    this.position.y = floor;
                    vy = 0;
                }

                if (this.position.y < 0) {
                    this.position.y = 0;
                    vy = 0;
                }

                _originalUpdate();
                this.velocity.y = 0;
            };
        };

        /* BUILDER_ONLY_START */
        try {
            setTimeout(() => {
                try {
                    const objs = Array.isArray(gameEnv?.gameObjects) ? gameEnv.gameObjects : [];
                    const summary = objs.map(o => ({ cls: o?.constructor?.name || 'Unknown', id: o?.canvas?.id || '', z: o?.canvas?.style?.zIndex || '' }));
                    if (window && window.parent) window.parent.postMessage({ type: 'rpg:objects', summary }, '*');
                } catch (_) {}
            }, 250);
        } catch (_) {}
        try {
            if (window && window.parent) {
                try {
                    const rect = (gameEnv && gameEnv.container && gameEnv.container.getBoundingClientRect) ? gameEnv.container.getBoundingClientRect() : { top: gameEnv.top || 0, left: 0 };
                    window.parent.postMessage({ type: 'rpg:env-metrics', top: rect.top, left: rect.left }, '*');
                } catch (_) {
                    try { window.parent.postMessage({ type: 'rpg:env-metrics', top: gameEnv.top, left: 0 }, '*'); } catch (__){ }
                }
            }
        } catch (_) {}
        try {
            window.addEventListener('message', (e) => {
                if (!e || !e.data) return;
                if (e.data.type === 'rpg:toggle-walls') {
                    const show = !!e.data.visible;
                    if (Array.isArray(gameEnv?.gameObjects)) {
                        for (const obj of gameEnv.gameObjects) {
                            if (obj instanceof Barrier) {
                                obj.visible = show;
                            }
                        }
                    }
                } else if (e.data.type === 'rpg:set-drawn-barriers') {
                    const arr = Array.isArray(e.data.barriers) ? e.data.barriers : [];
                    window.__overlayBarriers = window.__overlayBarriers || [];
                    try {
                        for (const ob of window.__overlayBarriers) {
                            if (ob && typeof ob.destroy === 'function') ob.destroy();
                        }
                    } catch (_) {}
                    window.__overlayBarriers = [];
                    for (const bd of arr) {
                        try {
                            const data = {
                                id: bd.id,
                                x: bd.x,
                                y: bd.y,
                                width: bd.width,
                                height: bd.height,
                                visible: !!bd.visible,
                                hitbox: { widthPercentage: 0.0, heightPercentage: 0.0 },
                                fromOverlay: true
                            };
                            const bobj = new Barrier(data, gameEnv);
                            gameEnv.gameObjects.push(bobj);
                            window.__overlayBarriers.push(bobj);
                        } catch (_) {}
                    }
                }
            });
        } catch (_) {}
        /* BUILDER_ONLY_END */
    }
}

export const gameLevelClasses = [GameLevelCustom];


This prevents the player from falling infinitely fast and breaking the game.


### Platform Collision

Platforms stop the player from falling forever and allow jumping again.


In [ ]:
%%js


let y = 320;
let vy = 5;
const platformTop = 300;
const playerHeight = 20;

if (y + playerHeight >= platformTop && vy >= 0) {
    y = platformTop - playerHeight;
    vy = 0;
    console.log("Landed on platform!");
}

console.log("Final y:", y);


In [ ]:
%%js

// GAME_RUNNER: Platforms

import GameEnvBackground from '/assets/js/GameEnginev1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1/essentials/Player.js';
import Npc from '/assets/js/GameEnginev1/essentials/Npc.js';
import Barrier from '/assets/js/GameEnginev1/essentials/Barrier.js';

class GameLevelCustom {
    constructor(gameEnv) {
        const path = gameEnv.path;
        const width = gameEnv.innerWidth;
        const height = gameEnv.innerHeight;

        const bgData = {
            name: "custom_bg",
            src: path + "/images/gamebuilder/bg/alien_planet.jpg",
            pixels: { height: 772, width: 1134 }
        };

        const playerData = {
            id: 'playerData',
            src: path + "/images/gamebuilder/sprites/astro.png",
            SCALE_FACTOR: 5,
            STEP_FACTOR: 1000,
            ANIMATION_RATE: 50,
            INIT_POSITION: { x: 100, y: 52 },
            pixels: { height: 770, width: 513 },
            orientation: { rows: 4, columns: 4 },
            down: { row: 0, start: 0, columns: 3 },
            downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI/16 },
            downLeft: { row: 0, start: 0, columns: 3, rotate: -Math.PI/16 },
            left: { row: 2, start: 0, columns: 3 },
            right: { row: 1, start: 0, columns: 3 },
            up: { row: 3, start: 0, columns: 3 },
            upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI/16 },
            upRight: { row: 3, start: 0, columns: 3, rotate: -Math.PI/16 },
            hitbox: { widthPercentage: 0, heightPercentage: 0 },
            keypress: { up: 87, left: 65, down: 83, right: 68 },
            GRAVITY: false
        };

        // Platform barrier — sits at 60% down the canvas (matches platformTop in gravity)
        const platformData = {
            id: 'platform',
            x: 0,
            y: height * 0.6,
            width: width,
            height: 20,
            visible: true,
            color: 'rgba(180, 120, 40, 0.9)',
            hitbox: { widthPercentage: 0.0, heightPercentage: 0.0 },
            fromOverlay: true
        };

        this.classes = [
            { class: GameEnvBackground, data: bgData },
            { class: Player, data: playerData },
            { class: Barrier, data: platformData }
        ];

        // ── Gravity — mirrors the code runner example exactly ─────────────────
        // let y = player.position.y  (the engine owns this)
        // let vy = 0
        // const gravity = 0.5
        // every frame: vy += gravity;  y += vy;
        this.initialize = () => {
            const player = gameEnv.gameObjects.find(obj => obj instanceof Player);
            if (!player) return;

            let vy = 0;              // same as code runner example
            const gravity = 0.5;    // same as code runner example

            // jump() — gives an instant upward kick, just like the code runner example
            function jump() {
                vy = -10;  // negative = upward on screen
            }

            const _originalUpdate = player.update.bind(player);

            player.update = function () {
                // W key triggers jump() on fresh press
                const wDown = this.pressedKeys?.[87];
                if (wDown && !this._wWasDown) {
                    jump();
                }
                this._wWasDown = !!wDown;

                // gravity pulls down, capped at maxFall (terminal velocity)
                const maxFall = 10;
                vy = Math.min(vy + gravity, maxFall);

                // y += vy  — apply to position, not velocity so Player.js can't override it
                this.velocity.y = 0;
                this.position.y += vy;

                // platform collision — same logic as code runner example
                // if y + playerHeight >= platformTop && vy >= 0 → land on platform
                const platformTop = gameEnv.innerHeight * 0.6; // platform sits at 60% down
                if (this.position.y + this.height >= platformTop && vy >= 0) {
                    this.position.y = platformTop - this.height;
                    vy = 0;
                }

                // floor clamp
                const floor = gameEnv.innerHeight - this.height;
                if (this.position.y >= floor) {
                    this.position.y = floor;
                    vy = 0;
                }

                // ceiling clamp
                if (this.position.y < 0) {
                    this.position.y = 0;
                    vy = 0;
                }

                // run original update for drawing, collision, and left/right movement
                _originalUpdate();

                // zero velocity.y again so Player.js move() doesn't undo gravity
                this.velocity.y = 0;
            };
        };

        /* BUILDER_ONLY_START */
        try {
            setTimeout(() => {
                try {
                    const objs = Array.isArray(gameEnv?.gameObjects) ? gameEnv.gameObjects : [];
                    const summary = objs.map(o => ({ cls: o?.constructor?.name || 'Unknown', id: o?.canvas?.id || '', z: o?.canvas?.style?.zIndex || '' }));
                    if (window && window.parent) window.parent.postMessage({ type: 'rpg:objects', summary }, '*');
                } catch (_) {}
            }, 250);
        } catch (_) {}
        try {
            if (window && window.parent) {
                try {
                    const rect = (gameEnv && gameEnv.container && gameEnv.container.getBoundingClientRect) ? gameEnv.container.getBoundingClientRect() : { top: gameEnv.top || 0, left: 0 };
                    window.parent.postMessage({ type: 'rpg:env-metrics', top: rect.top, left: rect.left }, '*');
                } catch (_) {
                    try { window.parent.postMessage({ type: 'rpg:env-metrics', top: gameEnv.top, left: 0 }, '*'); } catch (__){ }
                }
            }
        } catch (_) {}
        try {
            window.addEventListener('message', (e) => {
                if (!e || !e.data) return;
                if (e.data.type === 'rpg:toggle-walls') {
                    const show = !!e.data.visible;
                    if (Array.isArray(gameEnv?.gameObjects)) {
                        for (const obj of gameEnv.gameObjects) {
                            if (obj instanceof Barrier) {
                                obj.visible = show;
                            }
                        }
                    }
                } else if (e.data.type === 'rpg:set-drawn-barriers') {
                    const arr = Array.isArray(e.data.barriers) ? e.data.barriers : [];
                    window.__overlayBarriers = window.__overlayBarriers || [];
                    try {
                        for (const ob of window.__overlayBarriers) {
                            if (ob && typeof ob.destroy === 'function') ob.destroy();
                        }
                    } catch (_) {}
                    window.__overlayBarriers = [];
                    for (const bd of arr) {
                        try {
                            const data = {
                                id: bd.id,
                                x: bd.x,
                                y: bd.y,
                                width: bd.width,
                                height: bd.height,
                                visible: !!bd.visible,
                                hitbox: { widthPercentage: 0.0, heightPercentage: 0.0 },
                                fromOverlay: true
                            };
                            const bobj = new Barrier(data, gameEnv);
                            gameEnv.gameObjects.push(bobj);
                            window.__overlayBarriers.push(bobj);
                        } catch (_) {}
                    }
                }
            });
        } catch (_) {}
        /* BUILDER_ONLY_END */
    }
}

export const gameLevelClasses = [GameLevelCustom];

### Grounded Jumping

This makes it so you can only jump when on a platform or on the ground

In [ ]:
%%js 

let vy = 0;
let isGrounded = false;

function jump() {
    if (isGrounded) {
        vy = -10;
        isGrounded = false;
    }
}

player.update = function () {
    const wDown = this.pressedKeys?.[87];
    if (wDown && !this._wWasDown) {
        jump();
    }
    this._wWasDown = !!wDown;

    const maxFall = 10;
    vy = Math.min(vy + gravity, maxFall);

    this.velocity.y = 0;
    this.position.y += vy;

    isGrounded = false; // reset each frame

    const platformTop = gameEnv.innerHeight * 0.6;
    if (this.position.y + this.height >= platformTop && vy >= 0) {
        this.position.y = platformTop - this.height;
        vy = 0;
        isGrounded = true;
    }

    const floor = gameEnv.innerHeight - this.height;
    if (this.position.y >= floor) {
        this.position.y = floor;
        vy = 0;
        isGrounded = true;
    }

    _originalUpdate();
    this.velocity.y = 0;
};

In [ ]:
%%js

// GAME_RUNNER: Grounding

import GameEnvBackground from '/assets/js/GameEnginev1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1/essentials/Player.js';
import Npc from '/assets/js/GameEnginev1/essentials/Npc.js';
import Barrier from '/assets/js/GameEnginev1/essentials/Barrier.js';

class GameLevelCustom {
    constructor(gameEnv) {
        const path = gameEnv.path;
        const width = gameEnv.innerWidth;
        const height = gameEnv.innerHeight;

        const bgData = {
            name: "custom_bg",
            src: path + "/images/gamebuilder/bg/alien_planet.jpg",
            pixels: { height: 772, width: 1134 }
        };

        const playerData = {
            id: 'playerData',
            src: path + "/images/gamebuilder/sprites/astro.png",
            SCALE_FACTOR: 5,
            STEP_FACTOR: 1000,
            ANIMATION_RATE: 50,
            INIT_POSITION: { x: 100, y: 52 },
            pixels: { height: 770, width: 513 },
            orientation: { rows: 4, columns: 4 },
            down: { row: 0, start: 0, columns: 3 },
            downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI/16 },
            downLeft: { row: 0, start: 0, columns: 3, rotate: -Math.PI/16 },
            left: { row: 2, start: 0, columns: 3 },
            right: { row: 1, start: 0, columns: 3 },
            up: { row: 3, start: 0, columns: 3 },
            upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI/16 },
            upRight: { row: 3, start: 0, columns: 3, rotate: -Math.PI/16 },
            hitbox: { widthPercentage: 0, heightPercentage: 0 },
            keypress: { up: 87, left: 65, down: 83, right: 68 },
            GRAVITY: false
        };

        const platformData = {
            id: 'platform',
            x: 0,
            y: height * 0.6,
            width: width,
            height: 20,
            visible: true,
            color: 'rgba(180, 120, 40, 0.9)',
            hitbox: { widthPercentage: 0.0, heightPercentage: 0.0 },
            fromOverlay: true
        };

        this.classes = [
            { class: GameEnvBackground, data: bgData },
            { class: Player, data: playerData },
            { class: Barrier, data: platformData }
        ];

        // ── Gravity + Grounded Jumping ───────────────────────────────
        this.initialize = () => {
            const player = gameEnv.gameObjects.find(obj => obj instanceof Player);
            if (!player) return;

            let vy = 0;
            let isGrounded = false;
            const gravity = 0.5;

            function jump() {
                if (isGrounded) {
                    vy = -10;
                    isGrounded = false;
                }
            }

            const _originalUpdate = player.update.bind(player);

            player.update = function () {
                const wDown = this.pressedKeys?.[87];
                if (wDown && !this._wWasDown) {
                    jump();
                }
                this._wWasDown = !!wDown;

                const maxFall = 10;
                vy = Math.min(vy + gravity, maxFall);

                this.velocity.y = 0;
                this.position.y += vy;

                // assume airborne each frame
                isGrounded = false;

                // platform collision
                const platformTop = gameEnv.innerHeight * 0.6;
                if (this.position.y + this.height >= platformTop && vy >= 0) {
                    this.position.y = platformTop - this.height;
                    vy = 0;
                    isGrounded = true;
                }

                // floor collision
                const floor = gameEnv.innerHeight - this.height;
                if (this.position.y >= floor) {
                    this.position.y = floor;
                    vy = 0;
                    isGrounded = true;
                }

                // ceiling clamp
                if (this.position.y < 0) {
                    this.position.y = 0;
                    vy = 0;
                }

                _originalUpdate();
                this.velocity.y = 0;
            };
        };

        /* BUILDER_ONLY_START */
        try {
            setTimeout(() => {
                try {
                    const objs = Array.isArray(gameEnv?.gameObjects) ? gameEnv.gameObjects : [];
                    const summary = objs.map(o => ({ cls: o?.constructor?.name || 'Unknown', id: o?.canvas?.id || '', z: o?.canvas?.style?.zIndex || '' }));
                    if (window && window.parent) window.parent.postMessage({ type: 'rpg:objects', summary }, '*');
                } catch (_) {}
            }, 250);
        } catch (_) {}

        try {
            if (window && window.parent) {
                try {
                    const rect = (gameEnv && gameEnv.container && gameEnv.container.getBoundingClientRect) ? gameEnv.container.getBoundingClientRect() : { top: gameEnv.top || 0, left: 0 };
                    window.parent.postMessage({ type: 'rpg:env-metrics', top: rect.top, left: rect.left }, '*');
                } catch (_) {
                    try { window.parent.postMessage({ type: 'rpg:env-metrics', top: gameEnv.top, left: 0 }, '*'); } catch (__){}
                }
            }
        } catch (_) {}

        try {
            window.addEventListener('message', (e) => {
                if (!e || !e.data) return;

                if (e.data.type === 'rpg:toggle-walls') {
                    const show = !!e.data.visible;
                    if (Array.isArray(gameEnv?.gameObjects)) {
                        for (const obj of gameEnv.gameObjects) {
                            if (obj instanceof Barrier) {
                                obj.visible = show;
                            }
                        }
                    }
                }
            });
        } catch (_) {}
        /* BUILDER_ONLY_END */
    }
}

export const gameLevelClasses = [GameLevelCustom];

### Platformers in Games 🎮
Ex:
- In **Mario**: You jump and fall back down due to gravity  
- In **Minecraft**: You fall if there is no block under you  (except in our game there is no fall damage)
➡️ Gravity is always running in the background
